# QCoDeS in 15 mins: QTHz Lab version

## Imports

In [ ]:
import numpy as np
from scipy.io import savemat
import yaml, logging, time, json, datetime as dt
from pathlib import Path

import qcodes as qc
from qcodes.station import Station
from qcodes.parameters import Parameter
from qcodes.dataset import (
    Measurement,
    initialise_or_create_database_at,
    load_or_create_experiment
)
from qcodes.instrument import Instrument
from qcodes.instrument_drivers.stanford_research import SR860

## Setup logging

In [ ]:
log = logging.getLogger("tutorial")
log.setLevel(logging.INFO)
log.propagate = False
for h in list(log.handlers):          # clear stale handlers on re-run
    log.removeHandler(h)
    h = logging.StreamHandler(sys.stdout)
    h.setFormatter(logging.Formatter("%(asctime)s  %(levelname)-7s %(message)s", "%H:%M:%S"))
    log.addHandler(h)

qc.logger.start_all_logging()
log.info("QCoDeS log files in: %s", qc.logger.get_log_file_name())

## Read Configuration

In [ ]:
CONFIG_PATH = Path("config.yaml").resolve()
ROOT = CONFIG_PATH.parent
config_text = CONFIG_PATH.read_text()
cfg = yaml.safe_load(config_text)

log.info("loaded config: %s", 'path to config')


## Database and Experiment

In [ ]:
db_path = (ROOT / cfg["database"]).resolve()
db_path.parent.mkdir(parents=True, exist_ok=True)

initialise_or_create_database_at(db_path)
log.info("Database: %s (%.1f MB)", db_path, db_path.stat().st_size / 1e6)

exp = load_or_create_experiment(
    experiment_name = cfg["experiment_name"],
    sample_name = cfg['sample_name']
)
log.info("Experiment #%d '%s' on sample '%s' - %d run(s) so far", exp.exp_id, exp.name, exp.sample_name, len(exp.data_sets()))

## Instruments and Station

In [ ]:
Instrument.close_all()
station = Station()

lck_cfg = cfg['instruments']['lockin']

t0 = time.perf_counter()
lockin = SR860("lockin", lck_cfg['address'])
log.info("Connected to %s at %s in %.2f s",
         lockin.IDN(), lck_cfg['address'], time.perf_counter()-t0)

for pname, requested in lck_cfg.get("settings", {}).items():
    param = lockin.parameters[pname]
    param(requested)
    time.sleep(0.5)
    actual = param()
    flag = "" if actual == requested else "  <-- DIFFERS from requested %r" % requested
    log.info("%-16s = %-10r %s%s", pname, actual, param.unit, flag)

station.add_component(lockin)
log.info("Station components: %s", list(station.components))

## Register parameters

In [ ]:
elapsed_time = Parameter(
    "elapsed_time",
    label = "Time",
    unit = "s",
    set_cmd = None,
    get_cmd = None
)

meas = Measurement(
    exp = exp,
    station = station,
    name = "QCoDeS_tutorial"
)
meas.write_period = 1.0

meas.register_parameter(elapsed_time)
recorded = [lockin.X, lockin.Y, lockin.R, lockin.P]
for p in recorded:
    meas.register_parameter(p, setpoints = (elapsed_time,))

log.info("Registered %d columns:", len(meas.parameters))

## Measurement

In [ ]:
mcfg = cfg["measurement"]
duration = mcfg["duration"]
interval = mcfg["sample_interval"]
tc = lockin.time_constant()

time.sleep(3*tc)

with meas.run() as datasaver:
    ds = datasaver.dataset
    log.info("Run #%d started (guid %s)", ds.run_id, ds.guid)

    n = 0
    t_start = time.perf_counter()
    while True:
        t = time.perf_counter() - t_start
        if t > duration:
            break
        x, y = lockin.get_values("X", "Y")
        r, p = lockin.get_values("R", "P")

        datasaver. add_result(
            (elapsed_time, t),
            (lockin.X, x),
            (lockin.Y, y),
            (lockin.R, r),
            (lockin.P, p)
        )

        n += 1
        if n % max(1, int(1.0 / interval)) == 0:
            log.info("t = %6.2f s   X = %+.4e   Y=%+.4e", t, x, y)

        time.sleep(max(0.0, interval - ((time.perf_counter() - t_start) - t)))

log.info("Run #%d finished: %d points in %.2f s (%.1f Hz)", ds.run_id, n, t, n/t)

## Matlab data format

In [ ]:
export_dir = (ROOT / cfg["export_dir"]).resolve()
export_dir.mkdir(parents=True, exist_ok=True)

df = ds.to_pandas_dataframe().reset_index()
stamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
mat_path = export_dir / f"run{ds.run_id:04}_{ds.guid[:8]}_{stamp}.mat"

mat = {col: df[col].to_numpy() for col in df.columns}
mat.update({
    "run_id": ds.run_id,
    "guid": ds.guid,
    "experiment": exp.name,
    "sample": exp.sample_name,
    "config_yaml": config_text,
    "snapshot_json": json.dumps(ds.snapshot),
    "units": {p.name: p.unit for p in [elapsed_time, *recorded]}
})

savemat(mat_path, mat, do_compression=True)
log.info("MATLAB file: %s (%.1f kB, %d rows)",
         mat_path.name, mat_path.stat().st_size / 1e3, len(df))